In [6]:
%pip install networkx

/bin/bash: line 1: pipenv: command not found


In [7]:
import json
import difflib
from collections import deque, defaultdict
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

ModuleNotFoundError: No module named 'networkx'

In [4]:
# helpers to compute similarity between nodes in KG and causal graph

def normalize(s: str) -> str:
    return " ".join(s.lower().strip().split())

def token_set_ratio(a: str, b: str) -> float:
    set_a = set(normalize(a).split())
    set_b = set(normalize(b).split())
    if not set_a or not set_b:
        return 0.0
    intersect = len(set_a & set_b)
    union_len = len(set_a | set_b)
    return intersect / union_len


def similarity_score(a: str, b: str) -> float:
    # compute similarity score between two strings, output [0, 1s]
    ts = token_set_ratio(a, b)
    seq = difflib.SequenceMatcher(None, normalize(a), normalize(b)).ratio()
    return max(ts, seq)

In [5]:
def match_attributes_to_kg_nodes(attributes, kg_nodes, threshold=0.55):
    mappings = {}
    scores = {}

    kg_node_list = list(kg_nodes)
    for attr in attributes:
        best_score = 0.0
        best_node = None

        for node in kg_node_list:
            score = similarity_score(attr, node)
            if score > best_score:
                best_score = score
                best_node = node

        if best_score >= threshold:
            mappings[attr] = best_node
            scores[attr] = best_score
        else:
            mappings[attr] = None
            scores[attr] = best_score

    return mappings, scores

In [6]:
def bfs_paths(start, graph, blocked, max_depth=4):
    # pathfinding w/ BFS
    paths = []
    queue = deque([(start, [start])])

    while queue:
        node, path = queue.popleft()
        if len(path) > max_depth:
            continue

        for (nbr, edge_label) in graph.get(node, []):
            if nbr in path:
                continue

            new_path = path + [nbr]
            paths.append((new_path, edge_label))


            if nbr not in blocked:
                queue.append((nbr, new_path))

    return paths

In [7]:
def extract_causal_graph(causal_nodes, mappings, kg_graph):
    mapped_nodes = {v for v in mappings if v is not None}

    # output
    causal = {node: [] for node in causal_nodes}

    # get adj list from json
    adj = defaultdict(list)
    for src, edges in kg_graph.items():
        for e in edges:
            if "edge" in e and "neighbor" in e:
                adj[src].append((e["edge"], e["neighbor"]))

    # add any direct edges
    for A in mapped_nodes:
        for (edge, nbr) in adj[A]:
            if nbr in mapped_nodes:
                causal[mappings[A]].append({
                    "edge": edge,
                    "neighbor": mappings[nbr]
                })

    # add any composite edges (check if this works :/)
    for A in mapped_nodes:
        blocked = mapped_nodes - {A}

        for path, last_edge in bfs_paths(A, adj, blocked):
            if len(path) < 2:
                continue

            B = path[-1]
            if B not in mapped_nodes or B == A:
                continue

            # direct edge - alr added
            if len(path) == 2:
                continue

            # build composite predicate
            composite_pred = []
            for i in range(len(path) - 1):
                src = path[i]
                dst = path[i + 1]
                for (edge, nbr) in adj[src]:
                    if nbr == dst:
                        composite_pred.append(edge)
                        break

            causal[mappings[A]].append({
                "edge": "; ".join(composite_pred),
                "neighbor": mappings[B]
            })

    return causal


In [8]:
file_path = 'mortgage-q10-blade/knowledge-graph.json'
output_path = 'mortgage-q10-blade/causal-graph.json'
graph_path = 'mortgage-q10-blade/causal-graph.html'

In [9]:
try:
    with open(file_path, 'r') as file:
        graph_json = json.load(file)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except json.JSONDecodeError:
    print(f"Error: Failed to decode JSON from the file '{file_path}'. Check for malformed JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

attributes = [
           "female", "black", "housing expense ratio", "self employed",
             "married", "mortgage credit", "consumer credit", "bad history", 
             "PI ratio", "deny", "loan to value", "denied PMI", "accept"
        ]

mappings, scores = match_attributes_to_kg_nodes(attributes, graph_json)

causal_nodes = {k for k in mappings.keys()}

reverse_mappings = {v: k for k, v in mappings.items() if v is not None}

print(reverse_mappings)

causal_graph = extract_causal_graph(causal_nodes, reverse_mappings, graph_json)

print(mappings)

with open(output_path, 'w') as outfile:
    json.dump(causal_graph, outfile, indent=2)


{'Self-Employed (1099)': 'self employed', 'Credit History': 'bad history', 'Loan Type': 'loan to value'}
{'female': None, 'black': None, 'housing expense ratio': None, 'self employed': 'Self-Employed (1099)', 'married': None, 'mortgage credit': None, 'consumer credit': None, 'bad history': 'Credit History', 'PI ratio': None, 'deny': None, 'loan to value': 'Loan Type', 'denied PMI': None, 'accept': None}


In [10]:
# build graph -> NetworkX library
G = nx.DiGraph()

for node in causal_graph.keys():
    G.add_node(node)

for node, edges in causal_graph.items():
    for edge in edges:
        neighbor = edge.get("neighbor")
        relation = edge.get("edge", "")

        if neighbor:
            G.add_node(neighbor)
            G.add_edge(node, neighbor, label=relation)

# visualize graph
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5, iterations=50)
edge_labels = nx.get_edge_attributes(G, "label")

nx.draw_networkx_nodes(G, pos, node_size=700, node_color="#66b3ff")
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="->", arrowsize=15)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title("Causal Graph")
plt.axis("off")
plt.tight_layout()
plt.show()


NameError: name 'nx' is not defined

In [11]:
net = Network(height="750px", width="100%", directed=True, bgcolor="#ffffff", font_color="#000000")
net.from_nx(G)

net.repulsion(
    node_distance=150,
    central_gravity=0.33,
    spring_length=200,
    spring_strength=0.05,
    damping=0.95
)

for src, dst, data in G.edges(data=True):
    label = data.get("label", "")
    if label:
        net.edges[-1]["title"] = label

net.show(graph_path, notebook=False)

NameError: name 'Network' is not defined

# Testing with adversarial input

In [12]:
file_path = 'knowledge-graph-v2.json'

try:
    with open(file_path, 'r') as file:
        graph_json = json.load(file)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except json.JSONDecodeError:
    print(f"Error: Failed to decode JSON from the file '{file_path}'. Check for malformed JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

attributes = [
            "female", "black", "housing_expense_ratio", "self_employed",
             "married", "mortgage_credit", "consumer_credit", "bad_history", 
             "PI_ratio", "deny", "loan_to_value", "denied_PMI", "accept"
        ]

mappings, scores = match_attributes_to_kg_nodes(attributes, graph_json)

causal_nodes = {k for k in mappings.keys()}

reverse_mappings = {v: k for k, v in mappings.items() if v is not None}

print(reverse_mappings)

adversarial_causal_graph = extract_causal_graph(causal_nodes, reverse_mappings, graph_json)

print(mappings)

with open('causal-graph-adversarial.json', 'w') as outfile:
    json.dump(adversarial_causal_graph, outfile, indent=2)

adversarial_G = nx.DiGraph()

for node in adversarial_causal_graph.keys():
    adversarial_G.add_node(node)

for node, edges in adversarial_causal_graph.items():
    for edge in edges:
        neighbor = edge.get("neighbor")
        relation = edge.get("edge", "")

        if neighbor:
            adversarial_G.add_node(neighbor)
            adversarial_G.add_edge(node, neighbor, label=relation)

net = Network(height="750px", width="100%", directed=True, bgcolor="#ffffff", font_color="#000000")
net.from_nx(adversarial_G)

net.repulsion(
    node_distance=150,
    central_gravity=0.33,
    spring_length=200,
    spring_strength=0.05,
    damping=0.95
)

for src, dst, data in adversarial_G.edges(data=True):
    label = data.get("label", "")
    if label:
        net.edges[-1]["title"] = label

net.show("adversarial_causal_graph.html", notebook=False)

Error: The file 'knowledge-graph-v2.json' was not found.
{'Self-Employed (1099)': 'self_employed', 'Credit History': 'bad_history', 'Loan-to-Value Ratio (LTV)': 'loan_to_value'}
{'female': None, 'black': None, 'housing_expense_ratio': None, 'self_employed': 'Self-Employed (1099)', 'married': None, 'mortgage_credit': None, 'consumer_credit': None, 'bad_history': 'Credit History', 'PI_ratio': None, 'deny': None, 'loan_to_value': 'Loan-to-Value Ratio (LTV)', 'denied_PMI': None, 'accept': None}


NameError: name 'nx' is not defined